In [ ]:
from itertools import combinations

def build_sequences_generic(
    df, prep_fn, td_keep_cols,
    k_in=2, k_out=1,
    mode="consecutive",
    min_tests=None,
    min_span_years=0.0
):
    """
    Build variable-length visual field history sequences.

    Parameters
    ----------
    df : DataFrame
        Must contain PatID, Eye, FieldN, Time_from_Baseline and TD_* columns.
    prep_fn : function
        Converts one row into its TD vector (length L).
    td_keep_cols : list
        TD columns to keep (length L).
    k_in : int
        Number of input visits in history (T).
    k_out : int
        Number of future visits to predict (currently only k_out=1)
    mode : "consecutive" or "all"
    min_tests : int
        Minimum total tests required per eye (default = k_in + k_out)
    min_span_years : float
        Minimum time gap between first and last visit in the window.

    Returns
    -------
    X : ndarray, shape (N, k_in, L)
    Y : ndarray, shape (N, L)
    meta_df : DataFrame with sequence metadata.
    """
    assert k_out == 1, "Only k_out=1 supported currently."
    need = k_in + k_out
    if min_tests is None:
        min_tests = need

    df_sorted = df.sort_values(["PatID", "Eye", "FieldN"]).reset_index(drop=True)
    X_list, Y_list, meta = [], [], []
    L = len(td_keep_cols)

    for (pid, eye), g in df_sorted.groupby(["PatID", "Eye"]):
        g = g.sort_values("FieldN")
        if len(g) < min_tests:
            continue

        idx = list(range(len(g)))

        if mode == "consecutive":
            windows = [tuple(range(s, s+need)) for s in range(0, len(g)-need+1)]
        elif mode == "all":
            windows = combinations(idx, need)
        else:
            raise ValueError("mode must be 'consecutive' or 'all'")

        for win in windows:
            rows = [g.iloc[w] for w in win]
            times = [float(r["Time_from_Baseline"]) for r in rows]

            # Ensure chronological order and time span
            if not all(times[i] <= times[i+1] for i in range(len(times)-1)):
                continue
            if (times[-1] - times[0]) < min_span_years:
                continue

            # Build input (k_in) and target (1)
            inputs = rows[:k_in]
            target = rows[-1]

            vec_in = [prep_fn(r) for r in inputs]
            vec_out = prep_fn(target)
            if any(v is None for v in vec_in) or vec_out is None:
                continue

            X = np.stack(vec_in, axis=0)   # (k_in, L)
            Y = vec_out                   # (L,)
            X_list.append(X)
            Y_list.append(Y)
            meta.append({
                "PatID": pid,
                "Eye": eye,
                "FieldNs": [int(r["FieldN"]) for r in rows],
                "Times": times
            })

    if not X_list:
        return np.empty((0,k_in,L)), np.empty((0,L)), pd.DataFrame()

    X = np.stack(X_list)
    Y = np.stack(Y_list)
    meta_df = pd.DataFrame(meta)
    return X, Y, meta_df

In [ ]:
# k_in = 2: use 2 past VFs to predict the next VF
X2, Y2, M2 = build_sequences_generic(
    df=df,
    prep_fn=prepare_td_vector,
    td_keep_cols=td_keep,
    k_in=2,
    k_out=1,
    mode="consecutive",     
    min_span_years=0.0      
)
print("k_in=2: X2", X2.shape, "Y2", Y2.shape, "M2", M2.shape)

# k_in = 3: use 3 past VFs to predict the next VF
X3, Y3, M3 = build_sequences_generic(
    df=df,
    prep_fn=prepare_td_vector,
    td_keep_cols=td_keep,
    k_in=3,
    k_out=1,
    mode="consecutive",
    min_span_years=0.0
)
print("k_in=3: X3", X3.shape, "Y3", Y3.shape, "M3", M3.shape)

# k_in = 4: use 4 past VFs to predict the next VF
X4, Y4, M4 = build_sequences_generic(
    df=df,
    prep_fn=prepare_td_vector,
    td_keep_cols=td_keep,
    k_in=4,
    k_out=1,
    mode="consecutive",
    min_span_years=0.0
)
print("k_in=4: X4", X4.shape, "Y4", Y4.shape, "M4", M4.shape)

In [ ]:
X_hist_list = []
Y_list = []
time_hist_list = []

# 1) k_in = 2
for i in range(X2.shape[0]):
    X_hist_list.append(X2[i])      # (2, 52)
    Y_list.append(Y2[i])           # (52,)
    times = np.array(M2.iloc[i]["Times"], dtype=np.float32)   # length = 3 (2 history + 1 target)
    time_hist_list.append(times[:2])   # keep only history times

# 2) k_in = 3
for i in range(X3.shape[0]):
    X_hist_list.append(X3[i])      # (3, 52)
    Y_list.append(Y3[i])
    times = np.array(M3.iloc[i]["Times"], dtype=np.float32)   # length = 4
    time_hist_list.append(times[:3])

# 3) k_in = 4
for i in range(X4.shape[0]):
    X_hist_list.append(X4[i])      # (4, 52)
    Y_list.append(Y4[i])
    times = np.array(M4.iloc[i]["Times"], dtype=np.float32)   # length = 5
    time_hist_list.append(times[:4])

Y_array = np.stack(Y_list, axis=0)

print("Total sequences:", len(X_hist_list))
print("Y_array shape:", Y_array.shape)
print("Example sequence shapes/time:")
print("  X_hist_list[0].shape:", X_hist_list[0].shape)
print("  time_hist_list[0]:", time_hist_list[0])